# 02 · Embeddings and similarity

After this notebook you can explain what an embedding is, pick between cosine, dot product and Euclidean distance (and prove when they agree), show exactly where keyword vectors (TF-IDF) fail and neural embeddings win, build top-k semantic search in five lines, and reason about dimensions, normalisation and Matryoshka truncation.

**Time:** ~25 min · **Runs:** offline on CPU in < 1 min (uses the cached `all-MiniLM-L6-v2` model, 90 MB) · **Needs:** [01 · Tokenization](01_tokenization_bpe_and_tiktoken.ipynb), [NumPy for AI engineers](../04_numpy_pandas_pytorch/01_numpy_for_ai_engineers.ipynb)

## Why this matters in interviews

- Embeddings are the engine of RAG, semantic search, semantic caching, clustering, dedup and recommendation. "Why did retrieval miss this document?" is usually an embedding question.
- Interviewers probe the maths: cosine vs dot vs L2, why normalise, what dimensionality costs in memory.
- The honest comparison of lexical vs dense retrieval (and the traps: negation, exact IDs, domain shift) separates people who have shipped search from people who have read about it.
- Newer knobs - Matryoshka embeddings, OpenAI's `dimensions=` - come up in "how would you cut vector storage 4x?" questions.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `fo04` | What is an embedding? |
| `ve01` | What is cosine similarity, and what are the alternatives? |
| `ve02` | When would you use dot product instead of cosine? |
| `ve13` | Lexical versus dense retrieval — what is the honest comparison? |
| `ve15` | What are Matryoshka embeddings? |
| `ve20` | What is the curse of dimensionality and does it affect embeddings? |
| `ve23` | What is normalisation and when must you do it? |
| `ve31` | What is the difference between semantic similarity and relevance? |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
pd.set_option("display.width", 120)
print("ready")

## 1. What an embedding is

**In plain English:** an embedding model turns a piece of text into a list of numbers (a vector, typically 384-3072 numbers long) such that texts with similar meaning end up close together. "Close" is then a geometry question we can answer with fast linear algebra, over millions of documents.

Technically, a sentence embedding model is usually a small transformer encoder: tokens -> contextual token vectors -> *pooling* (mean over tokens) -> optional L2 normalisation. It is trained *contrastively*: pull (question, answer) or (sentence, paraphrase) pairs together, push random pairs apart. Nobody hand-designs the dimensions; "meaning" is spread across all of them.

Our running data: 20 sentences from 4 topics, and one paraphrased query per sentence written with *different words* on purpose.

In [ ]:
DOCS = [
    # sports
    "The striker scored twice in the second half to win the match.",
    "Our team lost the basketball game in overtime.",
    "She trains every morning for the city marathon.",
    "The goalkeeper saved a penalty in the final minute.",
    "Tennis fans packed the stadium for the championship final.",
    # cooking
    "Simmer the tomato sauce slowly with garlic and basil.",
    "Knead the dough for ten minutes before letting it rise.",
    "This curry recipe needs cumin, turmeric and fresh ginger.",
    "Roast the vegetables in a hot oven until golden.",
    "Whisk the eggs and sugar together for the cake batter.",
    # finance
    "The central bank raised interest rates to fight inflation.",
    "Stock prices fell sharply after the earnings report.",
    "I moved my savings into a low-cost index fund.",
    "The company issued bonds to pay down its debt.",
    "Mortgage payments rise when interest rates go up.",
    # tech
    "The new laptop has a faster processor and more memory.",
    "Update your phone's operating system to fix the security bug.",
    "The database query timed out under heavy load.",
    "We deployed the web app to the cloud with containers.",
    "The graphics card renders games at high frame rates.",
]
TOPICS = ["sports"] * 5 + ["cooking"] * 5 + ["finance"] * 5 + ["tech"] * 5
QUERIES = [  # QUERIES[i] is a paraphrase of DOCS[i]
    "Which forward netted two goals after half-time?",
    "We were beaten at hoops in extra time.",
    "Her daily dawn runs are preparation for a 42 km race.",
    "Keeper stops a spot kick at the very end.",
    "The arena was full of spectators for the title decider.",
    "How long should I cook a marinara on low heat?",
    "Bread making: work the mixture by hand, then leave it to prove.",
    "Which spices go into an Indian masala dish?",
    "Bake carrots and potatoes at high heat until they brown.",
    "Beat yolks with sweetener when making a sponge.",
    "Why did the Fed hike borrowing costs?",
    "Shares plunged when quarterly profits were announced.",
    "Is a cheap ETF tracking the market a good place for my money?",
    "The firm sold fixed-income securities to reduce what it owes.",
    "Why is my home loan getting more expensive?",
    "This notebook computer comes with a quicker CPU and extra RAM.",
    "Install the latest iOS patch to close the vulnerability.",
    "SQL requests are too slow when traffic spikes.",
    "We shipped our site to AWS using Docker.",
    "Which GPU gives smooth fps in video games?",
]
assert len(DOCS) == len(QUERIES) == len(TOPICS) == 20
print(f"{len(DOCS)} documents in {len(set(TOPICS))} topics, {len(QUERIES)} paraphrased queries")

## 2. Cosine vs dot product vs Euclidean distance

| Measure | Formula | Range | Cares about length? | Typical use |
|---|---|---|---|---|
| **Cosine similarity** | a·b / (‖a‖‖b‖) | -1 .. 1 | no, only direction | default for text embeddings |
| **Dot product** | a·b = Σ aᵢbᵢ | unbounded | yes | normalised embeddings (then = cosine, and cheaper); recommenders where length encodes popularity/quality |
| **Euclidean (L2)** | ‖a - b‖ | 0 .. ∞ | yes | k-means, some vector DB defaults |

**The key identity:** for unit vectors, ‖a - b‖² = 2 - 2·cos(a, b). So after L2 normalisation all three give *the same ranking*. Let us prove it numerically, then break it.

In [ ]:
docs_v = rng.normal(size=(8, 16))          # 8 random "documents" in 16 dims
query_v = rng.normal(size=16)

def l2_normalize(x):
    return x / np.linalg.norm(x, axis=-1, keepdims=True)

def rankings(docs, q):
    cos = l2_normalize(docs) @ l2_normalize(q)
    dot = docs @ q
    euc = np.linalg.norm(docs - q, axis=1)
    return np.argsort(-cos), np.argsort(-dot), np.argsort(euc)

u_docs, u_q = l2_normalize(docs_v), l2_normalize(query_v)
r_cos, r_dot, r_euc = rankings(u_docs, u_q)
print("unit vectors  -> cosine:", r_cos, " dot:", r_dot, " euclid:", r_euc)
assert (r_cos == r_dot).all() and (r_cos == r_euc).all()
assert np.allclose(np.linalg.norm(u_docs - u_q, axis=1) ** 2, 2 - 2 * (u_docs @ u_q))

long_i = r_cos[2]                          # the 3rd most similar doc ...
raw = u_docs.copy()
raw[long_i] *= 25.0                        # ... made 25x longer (same direction)
r_cos, r_dot, r_euc = rankings(raw, u_q)
print(f"doc {long_i} made 25x longer:")
print("               -> cosine:", r_cos, " dot:", r_dot, " euclid:", r_euc)
assert r_cos[2] == long_i, "cosine ignores length: the ranking is unchanged"
assert r_dot[0] == long_i, "dot product rewards length: the long doc jumps to first"
assert r_euc[-1] == long_i, "Euclidean punishes length: the long doc drops to last"

On unit vectors the three rankings are identical (asserted). Stretch one vector and the measures disagree: dot product rewards length, Euclidean punishes it, cosine ignores it. A 2-D picture makes the difference obvious:

In [ ]:
q2 = np.array([1.0, 0.25])
a2 = np.array([0.9, 0.3])        # same direction as the query, short
b2 = np.array([1.2, 2.6])        # different direction, long
fig, ax = plt.subplots(figsize=(5.5, 4.2))
for v, name, c in [(q2, "query", "black"), (a2, "doc A (close direction)", "tab:blue"), (b2, "doc B (long, off-direction)", "tab:red")]:
    ax.annotate("", xy=v, xytext=(0, 0), arrowprops=dict(arrowstyle="->", lw=2, color=c))
    cos_v = v @ q2 / (np.linalg.norm(v) * np.linalg.norm(q2))
    ax.text(v[0] + 0.04, v[1], f"{name}\ncos={cos_v:.2f}, dot={v @ q2:.2f}", color=c, fontsize=9)
ax.set_xlim(0, 2.6); ax.set_ylim(0, 3.0); ax.set_aspect("equal")
ax.set_title("Dot product prefers the long vector B; cosine prefers A")
ax.set_xlabel("dim 1"); ax.set_ylabel("dim 2"); ax.grid(alpha=0.3)
plt.show()

**Rule:** if your vectors are L2-normalised, use dot product (it *is* cosine, and it is the cheapest operation a vector index has). If they are not, decide whether length means something. For text similarity it usually does not, so normalise. In two-tower recommenders it often does (a popular item gets a longer vector), so maximum inner product search is deliberate there.

## 3. Keyword vectors (TF-IDF) vs neural sentence embeddings

TF-IDF represents a text by *which words it contains*, weighted by how rare each word is. It is the classic bag-of-words vector behind keyword search (BM25 is its better-tuned cousin). A neural embedding represents *what the text means*. Load the small, widely used `all-MiniLM-L6-v2` (384 dims):

In [ ]:
import transformers
transformers.logging.disable_progress_bar()          # keep the weight-loading bar out of the output
try:
    from sentence_transformers import SentenceTransformer, util
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
except Exception as e:
    raise RuntimeError("Could not load all-MiniLM-L6-v2 (a 90 MB one-time download from the Hugging Face Hub). "
                       f"Connect once to cache it, then re-run. Original error: {e}")
print(model)
print("embedding dimension:", model.get_embedding_dimension())

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

PAIRS = [
    ("paraphrase", "How do I reset my password?", "I forgot my login credentials and need new ones."),
    ("paraphrase", "The movie was fantastic.", "I really enjoyed the film."),
    ("paraphrase", "Where can I buy cheap flights?", "Looking for inexpensive airline tickets."),
    ("paraphrase", "A man is playing a guitar.", "Someone is strumming an instrument."),
    ("same word, other meaning", "The bank raised interest rates.", "We had a picnic on the river bank."),
    ("negation", "The drug is safe for children.", "The drug is not safe for children."),
]
tfidf = TfidfVectorizer().fit(DOCS + QUERIES + [t for _, a, b in PAIRS for t in (a, b)])

def tfidf_sim(a, b):
    va, vb = tfidf.transform([a, b]).toarray()      # rows are already L2-normalised
    return float(va @ vb)

def neural_sim(a, b):
    ea, eb = model.encode([a, b], normalize_embeddings=True)
    return float(ea @ eb)

pair_df = pd.DataFrame([(kind, a, b, tfidf_sim(a, b), neural_sim(a, b)) for kind, a, b in PAIRS],
                       columns=["kind", "text A", "text B", "TF-IDF", "MiniLM"])
print(pair_df.round(3).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
x = np.arange(len(pair_df))
ax.bar(x - 0.2, pair_df["TF-IDF"], 0.4, label="TF-IDF (words)")
ax.bar(x + 0.2, pair_df["MiniLM"], 0.4, label="MiniLM (meaning)")
ax.set_xticks(x, [f"{k}\n'{a[:18]}...'" for k, a in zip(pair_df["kind"], pair_df["text A"])], fontsize=8)
ax.set_ylabel("cosine similarity"); ax.set_ylim(0, 1.05)
ax.set_title("Paraphrases: TF-IDF sees no shared words, MiniLM sees shared meaning")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.show()

para = pair_df[pair_df["kind"] == "paraphrase"]
assert (para["MiniLM"] > para["TF-IDF"] + 0.3).all(), "neural embeddings should win every paraphrase pair"
bank = pair_df[pair_df["kind"].str.startswith("same")].iloc[0]
assert bank["MiniLM"] < para["MiniLM"].min(), "'bank' vs 'river bank' should score below every real paraphrase"
neg = pair_df[pair_df["kind"] == "negation"].iloc[0]
print(f"negation pair: MiniLM similarity {neg['MiniLM']:.2f} - near-identical, although the meanings are opposite")

Three lessons:

1. **Paraphrases:** TF-IDF scores ~0-0.13, because the only shared words are fillers like "I", "my", "a" ("cheap flights" vs "inexpensive airline tickets" shares nothing and scores exactly 0). MiniLM scores them 0.6-0.8. This is the whole reason dense retrieval exists.
2. **Ambiguity:** "bank" (money) vs "river bank" share a word, yet MiniLM scores the pair far below every real paraphrase: the vector depends on context, not on the word alone.
3. **The trap - negation:** "safe" vs "not safe" is ~0.9 similar for MiniLM, and TF-IDF is just as fooled. Embeddings capture *topic similarity*, not truth or logical relevance. That is why RAG systems add a reranker (cross-encoder) and why "semantic similarity is not relevance" is an interview favourite.

And the flip side (not shown here, see [BM25 from scratch](../09_rag/02_bm25_from_scratch.ipynb)): for exact identifiers - error codes, SKUs, names - keyword search beats embeddings. Production systems use both (hybrid search).

## 4. The similarity matrix: topics appear as blocks

In [ ]:
doc_emb = model.encode(DOCS, normalize_embeddings=True)
query_emb = model.encode(QUERIES, normalize_embeddings=True)
sim = doc_emb @ doc_emb.T
tfidf_docs = tfidf.transform(DOCS).toarray()
sim_tfidf = tfidf_docs @ tfidf_docs.T

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, m, title in [(axes[0], sim_tfidf, "TF-IDF cosine"), (axes[1], sim, "MiniLM cosine")]:
    shown = m.copy()
    np.fill_diagonal(shown, np.nan)                 # hide the trivial self-similarity of 1.0
    im = ax.imshow(shown, cmap="viridis", vmin=0, vmax=0.6)
    for b in [4.5, 9.5, 14.5]:
        ax.axhline(b, color="white", lw=1); ax.axvline(b, color="white", lw=1)
    ax.set_xticks([2, 7, 12, 17], ["sports", "cooking", "finance", "tech"])
    ax.set_yticks([2, 7, 12, 17], ["sports", "cooking", "finance", "tech"])
    ax.set_title(title)
fig.colorbar(im, ax=axes, shrink=0.8, label="cosine similarity")
plt.show()

def mean_block(m, same):
    t = np.array(TOPICS)
    mask = (t[:, None] == t[None, :]) if same else (t[:, None] != t[None, :])
    np.fill_diagonal(mask, False)
    return m[mask].mean()

print(f"MiniLM: mean within-topic {mean_block(sim, True):.2f} vs across-topic {mean_block(sim, False):.2f}")
print(f"TF-IDF: mean within-topic {mean_block(sim_tfidf, True):.2f} vs across-topic {mean_block(sim_tfidf, False):.2f}")
assert mean_block(sim, True) > mean_block(sim, False) + 0.1

MiniLM shows four brighter blocks on the diagonal: sentences about the same topic are closer to each other than to other topics, even with no shared words ("goalkeeper" and "marathon"). TF-IDF is almost all dark: different sentences rarely share words, so it cannot see topics at all. Note the absolute values: same-topic sentences average only ~0.2 here - "similar" is relative, which matters when you pick a threshold.

## 5. A 2-D map of meaning (PCA)

384 dimensions cannot be drawn, but PCA finds the 2 directions with the most variance. Circles are documents, crosses are their paraphrased queries.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=0).fit(doc_emb)
d2, q2d = pca.transform(doc_emb), pca.transform(query_emb)
colors = {"sports": "tab:blue", "cooking": "tab:orange", "finance": "tab:green", "tech": "tab:purple"}
fig, ax = plt.subplots(figsize=(9, 4.5))
for topic, c in colors.items():
    idx = [i for i, t in enumerate(TOPICS) if t == topic]
    ax.scatter(d2[idx, 0], d2[idx, 1], c=c, s=60, label=topic)
    ax.scatter(q2d[idx, 0], q2d[idx, 1], c=c, marker="x", s=50)
for i, text in enumerate(DOCS):
    ax.annotate(" ".join(text.split()[1:3]), d2[i], fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set_title(f"MiniLM embeddings projected to 2-D (PCA keeps {pca.explained_variance_ratio_.sum():.0%} of variance)")
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2"); ax.legend(fontsize=8)
plt.show()

nn_same_topic = [TOPICS[np.argsort(-sim[i])[1]] == TOPICS[i] for i in range(len(DOCS))]  # [0] is itself
print(f"nearest neighbour is in the same topic for {np.mean(nn_same_topic):.0%} of documents (in 384-D)")
assert np.mean(nn_same_topic) >= 0.8

Topics form separate clouds and the queries land in their topic's cloud. Two caveats interviewers like: a 2-D projection throws most of the information away (look at the variance kept in the title), so trust the 384-D numbers, not the picture; and t-SNE/UMAP plots can invent clusters, so never judge an embedding model from a pretty scatter plot.

## 6. Semantic search from scratch

Normalise once at index time, then search is one matrix-vector product and a top-k.

In [ ]:
def search(query, doc_matrix, docs, k=3):
    q = model.encode([query], normalize_embeddings=True)[0]
    scores = doc_matrix @ q                        # cosine, because both sides are unit length
    top = np.argpartition(-scores, k)[:k]          # O(n) partial selection, then sort only k items
    top = top[np.argsort(-scores[top])]
    return [(int(i), float(scores[i]), docs[i]) for i in top]

for q in ["my pasta sauce recipe", "rising borrowing costs", "phone security patch"]:
    print(f"\nquery: {q!r}")
    for i, s, d in search(q, doc_emb, DOCS):
        print(f"  {s:.3f}  [{i:2d}] {d}")

lib = util.semantic_search(model.encode(["rising borrowing costs"], normalize_embeddings=True), doc_emb, top_k=3)[0]
ours = [i for i, _, _ in search("rising borrowing costs", doc_emb, DOCS)]
assert ours == [hit["corpus_id"] for hit in lib], "our top-k must match sentence_transformers.util.semantic_search"
print("\nour search == sentence_transformers.util.semantic_search: same ids, same order")

That is exact (brute-force) nearest-neighbour search: perfect recall, O(N·d) per query. It is fine up to ~1M vectors on one machine; beyond that you use an approximate index (HNSW, IVF) - see [vector search with FAISS and HNSW](../09_rag/03_vector_search_faiss_hnsw.ipynb).

### Retrieval quality: does the right document come first?

For each paraphrased query we know the correct document, so we can measure **recall@1** (is it ranked first?) and **MRR** (mean of 1/rank).

In [ ]:
def retrieval_metrics(q_mat, d_mat):
    scores = q_mat @ d_mat.T
    ranks = np.array([int(np.where(np.argsort(-scores[i]) == i)[0][0]) + 1 for i in range(len(q_mat))])
    return float(np.mean(ranks == 1)), float(np.mean(1.0 / ranks))

tfidf_q = tfidf.transform(QUERIES).toarray()
results = {"TF-IDF": retrieval_metrics(tfidf_q, tfidf_docs), "MiniLM": retrieval_metrics(query_emb, doc_emb)}
for name, (r1, mrr) in results.items():
    print(f"{name:7s} recall@1 = {r1:.2f}   MRR = {mrr:.2f}")
assert results["MiniLM"][0] > results["TF-IDF"][0]

## 7. Embeddings through the API (`llm_setup.embed`)

In production you often call a hosted embedding model instead (OpenAI `text-embedding-3-small`, 1536 dims, or Ollama's `nomic-embed-text` locally). The code is the same for every provider.

> **Offline note:** without an API key or Ollama, `llm_setup` answers with a deterministic stand-in whose embeddings are *hashed bag-of-words with a small synonym table*. It is lexical, like TF-IDF: it knows car ≈ automobile from its table, but it cannot understand paraphrases. With a real key the same cells return real semantic vectors, and the recall numbers below typically jump well above TF-IDF's.

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER, embed

api_docs = l2_normalize(np.array(embed(DOCS)))          # 1 request for all 20 docs (batch your calls!)
api_queries = l2_normalize(np.array(embed(QUERIES)))    # 1 request
print(f"provider={PROVIDER}, model={EMBED_MODEL}, vectors: {api_docs.shape}")
results[f"API ({PROVIDER})"] = retrieval_metrics(api_queries, api_docs)
car, auto, banana = l2_normalize(np.array(embed(["car", "automobile", "banana"])))
print(f"cos(car, automobile) = {car @ auto:.2f}   cos(car, banana) = {car @ banana:.2f}")
for name, (r1, mrr) in results.items():
    print(f"{name:15s} recall@1 = {r1:.2f}   MRR = {mrr:.2f}")

### Fewer dimensions: OpenAI's `dimensions=` and Matryoshka embeddings

`text-embedding-3-*` models were trained with **Matryoshka Representation Learning (MRL)**: the loss is applied to the first 64, 128, 256, ... dimensions at once, so the *prefix* of the vector is itself a good embedding. `dimensions=256` returns that prefix, re-normalised, and cuts storage 6x. You can do the same yourself: slice and re-normalise.

In [ ]:
try:
    short = client.embeddings.create(model=EMBED_MODEL, input=["How do I reset my password?"], dimensions=256)
    v = np.array(short.data[0].embedding)
    print(f"dimensions=256 -> vector of length {len(v)}, norm {np.linalg.norm(v):.3f}")
except Exception as e:  # e.g. an Ollama model that does not support `dimensions`
    print(f"this provider/model does not support dimensions= ({type(e).__name__}); slice and re-normalise instead")

MiniLM was **not** trained with MRL, so we can measure how a normal model degrades when truncated. For an MRL model the curve stays flat much longer.

In [ ]:
dims_list = [8, 16, 32, 64, 128, 256, 384]
trunc = []
for d in dims_list:
    r1, mrr = retrieval_metrics(l2_normalize(query_emb[:, :d]), l2_normalize(doc_emb[:, :d]))
    trunc.append((d, r1, mrr, d * 4))
trunc_df = pd.DataFrame(trunc, columns=["dims", "recall@1", "MRR", "bytes/vector (fp32)"])
print(trunc_df.round(2).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(trunc_df["dims"], trunc_df["recall@1"], "o-", label="recall@1")
ax.plot(trunc_df["dims"], trunc_df["MRR"], "s--", label="MRR")
ax.axhline(results["TF-IDF"][0], color="gray", ls=":", label="TF-IDF recall@1")
ax.set_xscale("log", base=2); ax.set_xticks(dims_list, dims_list)
ax.set_xlabel("dimensions kept (prefix of the 384-d MiniLM vector)"); ax.set_ylabel("score")
ax.set_ylim(0, 1.05); ax.set_title("Truncating a non-Matryoshka embedding: quality falls as dims shrink")
ax.legend(); ax.grid(alpha=0.3)
plt.show()
assert trunc_df["MRR"].iloc[-1] >= trunc_df["MRR"].iloc[0]

Read it as a storage/quality trade-off: every halving of dimensions halves memory and search cost. Here recall@1 stays perfect down to 256 dims, is still high at 128, and collapses below 32. With only 20 queries each point is noisy - on real work you run this curve on *your own* labelled queries before choosing a dimension.

## 8. Normalisation and what dimensions cost

In [ ]:
print("MiniLM norms      :", np.linalg.norm(doc_emb[:4], axis=1).round(4), "(its last module is Normalize)")
print("TF-IDF row norms  :", np.linalg.norm(tfidf_docs[:4], axis=1).round(4), "(TfidfVectorizer norm='l2' default)")
unnorm = model.encode(DOCS[:4], normalize_embeddings=False)
print("encode(normalize_embeddings=False) is still unit-length here:", np.allclose(np.linalg.norm(unnorm, axis=1), 1))

n_docs = 1_000_000
rows = []
for d in [384, 768, 1536, 3072]:
    rows.append((d, n_docs * d * 4 / 1e9, n_docs * d * 2 / 1e9, n_docs * d * 1 / 1e9, n_docs * d / 8 / 1e9))
print(f"\nraw vector storage for {n_docs:,} documents (GB, before any index overhead):")
print(pd.DataFrame(rows, columns=["dims", "float32", "float16", "int8", "binary (1 bit)"]).round(3).to_string(index=False))

- **Normalise when** the model does not do it for you and you want cosine semantics from a dot-product index. Check with one `np.linalg.norm` call - models differ (MiniLM normalises inside the model; many others do not).
- **Never mix** vectors from two different models (or model versions) in one index: their spaces are unrelated. Re-embed everything when you change models.
- **Memory is linear** in dims x docs: 1M x 1536 float32 is ~6 GB before the index. Levers: fewer dims (MRL/`dimensions=`), float16/int8 (small loss), binary quantisation (32x smaller, use it to shortlist then rescore).

## 9. Anisotropy and hubness (briefly)

In [ ]:
t = np.array(TOPICS)
iu = np.triu_indices(len(DOCS), k=1)
same = (t[:, None] == t[None, :])[iu]

# Simulate an anisotropic model: every vector shares one big common direction.
common = 1.2 * l2_normalize(rng.normal(size=doc_emb.shape[1]))
aniso = l2_normalize(doc_emb + common)
centered = l2_normalize(aniso - aniso.mean(axis=0))            # the classic fix: subtract the mean vector
variants = {"MiniLM as is": doc_emb, "simulated anisotropic model": aniso, "after mean-centering": centered}

fig, ax = plt.subplots(figsize=(9, 3.6))
for name, e in variants.items():
    s = (e @ e.T)[iu]
    ax.hist(s[~same], bins=20, alpha=0.6, label=f"{name} (mean {s[~same].mean():.2f})")
ax.set_xlabel("cosine similarity of UNRELATED (different-topic) pairs"); ax.set_ylabel("pairs")
ax.set_title("A shared direction pushes every cosine up; centering removes it")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.show()
s_iso, s_an = sim[iu], (aniso @ aniso.T)[iu]
THRESH = 0.3                                                    # "related if cosine > 0.3", tuned on MiniLM
for name, s in [("MiniLM", s_iso), ("anisotropic", s_an)]:
    print(f"{name:12s} unrelated mean {s[~same].mean():.2f} | same-topic mean {s[same].mean():.2f} | "
          f"unrelated pairs above {THRESH}: {np.mean(s[~same] > THRESH):.0%}")
assert s_an[~same].mean() > s_iso[~same].mean() + 0.3
assert np.mean(s_an[~same] > THRESH) > np.mean(s_iso[~same] > THRESH), "the same threshold means something else"

in_top3 = np.bincount(np.argsort(-(query_emb @ doc_emb.T), axis=1)[:, :3].ravel(), minlength=len(DOCS))
print(f"hubness: how often each doc appears in some query's top-3 -> min {in_top3.min()}, max {in_top3.max()} "
      f"(uniform would be 3 each); most frequent: {DOCS[int(np.argmax(in_top3))]!r}")

- **Anisotropy:** MiniLM (contrastively trained) spreads its vectors well, so unrelated sentences sit near cosine 0. Embeddings from many other models do not: they share a dominant common direction and sit in a narrow cone, so even unrelated texts score 0.3-0.9 (the simulated middle histogram). Consequence: a fixed threshold ("similar if > 0.8") does not transfer between models. Calibrate thresholds on labelled pairs; mean-centering or whitening helps such models.
- **Hubness:** in high dimensions some points become nearest neighbours of *many* queries ("hubs") and pollute results. It gets worse with dimension and with anisotropy. Mitigations: normalisation, centering, rerankers, and MMR for diversity.

## Try it yourself

**1.** Add a fifth topic (say, "travel") with two documents and two paraphrased queries. Does MiniLM retrieve them correctly? Does TF-IDF?

In [ ]:
# Solution — try it yourself first, then run this
extra_docs = ["Our flight to Lisbon was delayed by three hours.", "Pack light and book the hostel in advance."]
extra_q = ["The plane to Portugal left late.", "Reserve cheap accommodation early and carry a small bag."]
d_all = model.encode(DOCS + extra_docs, normalize_embeddings=True)
q_new = model.encode(extra_q, normalize_embeddings=True)
top1 = np.argmax(q_new @ d_all.T, axis=1)
print("MiniLM top-1:", [(DOCS + extra_docs)[i] for i in top1])
assert list(top1) == [20, 21], "MiniLM should find both new travel documents"
tf = TfidfVectorizer().fit(DOCS + extra_docs + extra_q)
top1_tf = np.argmax(tf.transform(extra_q).toarray() @ tf.transform(DOCS + extra_docs).toarray().T, axis=1)
print("TF-IDF top-1:", [(DOCS + extra_docs)[i] for i in top1_tf])

**2.** Show numerically that for unit vectors, sorting by Euclidean distance ascending equals sorting by cosine descending, using the MiniLM query "rising borrowing costs".

In [ ]:
# Solution — try it yourself first, then run this
q = model.encode(["rising borrowing costs"], normalize_embeddings=True)[0]
by_cos = np.argsort(-(doc_emb @ q))
by_l2 = np.argsort(np.linalg.norm(doc_emb - q, axis=1))
print("cosine order:", by_cos[:5], " L2 order:", by_l2[:5])
assert (by_cos == by_l2).all()

**3.** Instead of truncating, reduce MiniLM to 16 dimensions with PCA fitted on the documents. Is it better or worse than keeping the first 16 dimensions?

In [ ]:
# Solution — try it yourself first, then run this
p16 = PCA(n_components=16, random_state=0).fit(doc_emb)
r_pca = retrieval_metrics(l2_normalize(p16.transform(query_emb)), l2_normalize(p16.transform(doc_emb)))
r_cut = retrieval_metrics(l2_normalize(query_emb[:, :16]), l2_normalize(doc_emb[:, :16]))
print(f"16 dims  PCA: recall@1={r_pca[0]:.2f} MRR={r_pca[1]:.2f}   |   first-16 slice: recall@1={r_cut[0]:.2f} MRR={r_cut[1]:.2f}")
print("(PCA on only 20 docs overfits them; with real data fit it on a large sample of your corpus)")

## Say it in an interview

- **30 seconds:** "An embedding is a vector from a model trained so that similar meanings land close together. Sentence embedders are transformer encoders with mean pooling, trained contrastively on pairs. You store document vectors, embed the query, and rank by cosine - which on L2-normalised vectors is just a dot product, and ranks identically to Euclidean distance because ‖a-b‖² = 2 - 2cos."
- **Lexical vs dense, honestly:** dense wins on paraphrase and synonyms; lexical wins on exact identifiers, rare names, codes and out-of-domain jargon, and is cheaper and explainable. Use hybrid search plus a reranker.
- **Traps:** embeddings encode topic, not truth (negation scores ~0.9); thresholds are model-specific (anisotropy); never mix models in one index; check whether your model already normalises.
- **Numbers:** MiniLM 384 dims; `text-embedding-3-small` 1536 (`-large` 3072), both accept `dimensions=`; 1M x 1536 float32 ≈ 6 GB raw.
- **Matryoshka:** trained so prefixes are good embeddings; truncate + re-normalise to trade a little quality for large memory/latency savings; a non-MRL model degrades faster (our curve).
- **Follow-ups:** "How would you evaluate an embedding model for your domain?" (labelled query-doc pairs, recall@k/MRR/nDCG on your data, not MTEB alone). "Semantic similarity vs relevance?" (a reranker or cross-encoder judges the query-document *pair*).

## Next

- [03 · Attention from scratch](03_attention_from_scratch.ipynb) - how the model makes each token's vector depend on its context.
- [Sentence-transformers embeddings and cross-encoders](../10_huggingface/04_sentence_transformers_embeddings_and_cross_encoders.ipynb)
- [Vector search with FAISS and HNSW](../09_rag/03_vector_search_faiss_hnsw.ipynb) · [BM25 from scratch](../09_rag/02_bm25_from_scratch.ipynb) · [Hybrid search and RRF](../09_rag/04_hybrid_search_and_rrf.ipynb) · [Reranking](../09_rag/05_reranking.ipynb)
- [Semantic search service (coding round)](../17_interview_coding_rounds/04_semantic_search_service.ipynb)
- Theory: [genai_flow course](../../genai_flow/index.html) (chapter 3 "Embeddings") · [interview bank](../../ai_interview_prep/index.html)